In [1]:
from pathlib import Path

import polars as pl

DATA_PATH = Path("../data/processed/importaciones_2019-2026_decoded.parquet")
EXPORT_DIR = Path("../outputs/tables")


def query_importaciones(
    years, importador: str | None = None, ncm: str | list[str] | None = None
) -> pl.DataFrame:
    """
    Query the reshaped imports dataset.

    Adds two columns derived from FECHA (a YYYYMM string): AÑO (year) and MES (month).
    """
    if importador is None and ncm is None:
        raise ValueError("Provide at least one of `importador` or `ncm`.")

    years_list = [years] if isinstance(years, int) else list(years)
    year_strs = [str(y) for y in years_list]

    filters = [pl.col("FECHA").str.slice(0, 4).is_in(year_strs)]

    if importador is not None:
        filters.append(pl.col("IMPORTADOR").str.contains(f"(?i){importador}"))

    if ncm is not None:
        ncm_list = [ncm] if isinstance(ncm, str) else list(ncm)
        ncm_filter = pl.col("NCM").str.starts_with(ncm_list[0])
        for code in ncm_list[1:]:
            ncm_filter = ncm_filter | pl.col("NCM").str.starts_with(code)
        filters.append(ncm_filter)

    combined_filter = filters[0]
    for f in filters[1:]:
        combined_filter = combined_filter & f

    lazy_df = pl.scan_parquet(DATA_PATH)
    lazy_df = lazy_df.filter(combined_filter).with_columns(
        pl.col("FECHA").str.slice(0, 4).cast(pl.Int64).alias("AÑO"),
        pl.col("FECHA").str.slice(4, 2).cast(pl.Int64).alias("MES"),
    )
    return lazy_df.collect()


def export_query_result(
    df: pl.DataFrame,
    query_years,
    query_importador: str | None,
    query_ncm: str | list[str] | None,
):
    """
    Dynamically generates a filename and exports the DataFrame to Excel.
    """
    if df.height == 0:
        print("The query returned 0 rows. No Excel file was created.")
        return

    # Extract start and end years
    years_list = [query_years] if isinstance(query_years, int) else list(query_years)
    start_year = min(years_list)
    end_year = max(years_list)
    year_suffix = (
        f"{start_year}-{end_year}" if start_year != end_year else str(start_year)
    )

    # Build dynamic prefix
    name_parts = []
    if query_importador:
        name_parts.append(query_importador.strip().replace(" ", "_").lower())
    if query_ncm:
        if isinstance(query_ncm, list):
            name_parts.append("-".join(query_ncm))
        else:
            name_parts.append(str(query_ncm))

    prefix = "_".join(name_parts)

    # Export
    export_filename = f"{prefix}_{year_suffix}.xlsx"
    export_path = EXPORT_DIR / export_filename

    print(f"Exporting {df.shape[0]} rows to: {export_path}")
    df.write_excel(export_path)
    print("Export complete!")

In [2]:
# --- Configuration for Query ---
query_years = [2022, 2023, 2024, 2025, 2026]
query_importador = "byrna argentina"
query_ncm = None  # e.g., "73" or ["73", "84"]

# --- Execute Query and Export ---
query_result = query_importaciones(
    years=query_years, importador=query_importador, ncm=query_ncm
)

# Export the results to Excel
export_query_result(query_result, query_years, query_importador, query_ncm)

# Display the first few rows
query_result.head()

Exporting 123 rows to: ..\outputs\tables\byrna_argentina_2022-2026.xlsx
Export complete!


ADUANA,DESTINACION,NUM_ITEM,FECHA,IMPORTADOR,MEDIO_TRANSPORTE,UNIDAD_MEDIDA,CANTIDAD_UNIDAD_MEDIDA,FOB_UNITARIO_USD,FOB_TOTAL_USD,DIVISA,PAIS_ORIGEN,PAIS_PROCEDENCIA,NCM,MONTO_TRIBUTADO_TOTAL,AÑO,MES
str,str,str,str,str,str,str,f64,f64,f64,str,str,str,str,f64,i64,i64
"""EZEIZA""","""24073IC04056012Z""","""3""","""202405""","""BYRNA ARGENTINA SA""","""AVION""","""UNIDAD""",6000.0,1110.0,36960.0,"""DOLAR ESTADOUNIDENSE""","""ESTADOS UNIDOS""","""ESTADOS UNIDOS""","""9305.99.00""",1008.15,2024,5
"""EZEIZA""","""24073IC04056012Z""","""4""","""202405""","""BYRNA ARGENTINA SA""","""AVION""","""UNIDAD""",3000.0,19080.0,36960.0,"""DOLAR ESTADOUNIDENSE""","""TAIWAN""","""ESTADOS UNIDOS""","""9305.99.00""",17328.97,2024,5
"""EZEIZA""","""24073IC04104089B""","""2""","""202409""","""BYRNA ARGENTINA SA""","""AVION""","""UNIDAD""",9000.0,25806.0,56062.5,"""DOLAR ESTADOUNIDENSE""","""CHINA""","""ESTADOS UNIDOS""","""9305.99.00""",22923.95,2024,9
"""EZEIZA""","""24073IC04141896X""","""2""","""202412""","""BYRNA ARGENTINA SA""","""AVION""","""UNIDAD""",6000.0,760.0,64491.99,"""DOLAR ESTADOUNIDENSE""","""SUDAFRICA""","""SUDAFRICA""","""7318.24.00""",602.18,2024,12
"""EZEIZA""","""24073IC04112396B""","""2""","""202410""","""BYRNA ARGENTINA SA""","""AVION""","""UNIDAD""",1735.0,62460.0,206635.98,"""DOLAR ESTADOUNIDENSE""","""SUDAFRICA""","""SUDAFRICA""","""9306.90.90""",58858.41,2024,10


In [3]:
# --- Configuration for Query ---
query_years = [2020, 2021, 2022, 2023, 2024, 2025, 2026]
query_importador = "bersa sociedad anonima"
query_ncm = None  # e.g., "73" or ["73", "84"]

# --- Execute Query and Export ---
query_result = query_importaciones(
    years=query_years, importador=query_importador, ncm=query_ncm
)

# Export the results to Excel
export_query_result(query_result, query_years, query_importador, query_ncm)

# Display the first few rows
query_result.head()

Exporting 767 rows to: ..\outputs\tables\bersa_sociedad_anonima_2020-2026.xlsx
Export complete!


ADUANA,DESTINACION,NUM_ITEM,FECHA,IMPORTADOR,MEDIO_TRANSPORTE,UNIDAD_MEDIDA,CANTIDAD_UNIDAD_MEDIDA,FOB_UNITARIO_USD,FOB_TOTAL_USD,DIVISA,PAIS_ORIGEN,PAIS_PROCEDENCIA,NCM,MONTO_TRIBUTADO_TOTAL,AÑO,MES
str,str,str,str,str,str,str,f64,f64,f64,str,str,str,str,f64,i64,i64
"""EZEIZA""","""20073IC04038523T""","""1""","""202004""","""BERSA SOCIEDAD ANONIMA""","""AVION""","""LITRO""",15.13,1356.0,1356.0,"""DOLAR ESTADOUNIDENSE""","""ESTADOS UNIDOS""","""ESTADOS UNIDOS""","""3208.90.39""",879.67,2020,4
"""BS.AS.(CAPITAL)""","""20001IC04079632Z""","""1""","""202005""","""BERSA SOCIEDAD ANONIMA""","""CAMION""","""UNIDAD""",2.0,3720.71,3384.0,"""EURO""","""ITALIA""","""ITALIA""","""8466.93.20""",1448.77,2020,5
"""BS.AS.(CAPITAL)""","""20001IC06005437K""","""1""","""202006""","""BERSA SOCIEDAD ANONIMA""",null,"""KILOGRAMO""",4428.0,21729.01,37951.34,"""EURO""","""ITALIA""","""ITALIA""","""7228.70.00""",11771.8,2020,6
"""BS.AS.(CAPITAL)""","""20001IC06007192K""","""1""","""202007""","""BERSA SOCIEDAD ANONIMA""",null,"""KILOGRAMO""",3965.0,6741.79,5888.03,"""EURO""","""ITALIA""","""ITALIA""","""7228.50.00""",5162.91,2020,7
"""BS.AS.(CAPITAL)""","""20001IC06008316J""","""2""","""202008""","""BERSA SOCIEDAD ANONIMA""",null,"""KILOGRAMO""",12140.0,21335.15,21505.56,"""EURO""","""ITALIA""","""ITALIA""","""7228.50.00""",11278.84,2020,8


In [ ]:
# --- Configuration for Query ---
query_years = [2020, 2021, 2022, 2023, 2024, 2025, 2026]
query_importador = None
query_ncm = "2701"

# --- Execute Query and Export ---
query_result = query_importaciones(
    years=query_years, importador=query_importador, ncm=query_ncm
)

# Export the results to Excel
export_query_result(query_result, query_years, query_importador, query_ncm)

# Display the first few rows
query_result.head()

Exporting 215 rows to: ..\outputs\tables\2701_2023-2026.xlsx
Export complete!


ADUANA,DESTINACION,NUM_ITEM,FECHA,IMPORTADOR,MEDIO_TRANSPORTE,UNIDAD_MEDIDA,CANTIDAD_UNIDAD_MEDIDA,FOB_UNITARIO_USD,FOB_TOTAL_USD,DIVISA,PAIS_ORIGEN,PAIS_PROCEDENCIA,NCM,MONTO_TRIBUTADO_TOTAL
str,str,str,str,str,str,str,f64,f64,f64,str,str,str,str,f64
"""SAN NICOLAS""","""26059IC65000037C""","""1""","""202603""","""AES ARGENTINA GENERACION SOCIE""","""ACUATICO""","""KILOGRAMO""",3.262799e7,2490168.2,2490168.2,"""DOLAR ESTADOUNIDENSE""","""COLOMBIA""","""COLOMBIA""","""2701.12.00""",2.2284e6
"""SAN NICOLAS""","""26059IC04000339A""","""1""","""202606""","""TERNIUM ARGENTINA S A""","""ACUATICO""","""KILOGRAMO""",2.2e7,5.0316e6,5.0316e6,"""DOLAR ESTADOUNIDENSE""","""AUSTRALIA""","""AUSTRALIA""","""2701.12.00""",2.8345e6
"""SAN NICOLAS""","""26059IC65000273E""","""1""","""202607""","""AES ARGENTINA GENERACION SOCIE""","""ACUATICO""","""KILOGRAMO""",4.1036e7,3217222.4,3217222.4,"""DOLAR ESTADOUNIDENSE""","""COLOMBIA""","""COLOMBIA""","""2701.12.00""",2.9234e6
"""CORDOBA""","""25017IC03000018J""","""1""","""202501""","""VULCANO SA""","""CAMION""","""KILOGRAMO""",44000.0,19440.0,19440.0,"""DOLAR ESTADOUNIDENSE""","""CHINA""","""CHINA""","""2701.11.00""",15330.15
"""BS.AS.(CAPITAL)""","""25001IC04010628L""","""1""","""202501""","""INQUINAT SA""","""ACUATICO""","""KILOGRAMO""",27000.0,15175.0,15175.0,"""DOLAR ESTADOUNIDENSE""","""CHINA""","""CHINA""","""2701.11.00""",10556.88
